# 🎯 실습: CLI 페르소나 챗봇 만들기

## 📋 실습 흐름

실무에서 챗봇을 처음 만들 때 거치는 5단계를 그대로 따라갑니다.

```
Step 0. 초기 설정          ← 패키지 설치, API 키 설정 (Colab/로컬 공통)
Step 1. 환경 점검          ← 1·2교시 코드가 정상 동작하는지 확인
Step 2. 페르소나 설계       ← 1단계+2단계 (요구사항·system 메시지)
Step 3. 챗봇 함수 만들기    ← 3단계 (MVP, prompt | llm)
Step 4. 명령어 처리        ← 4단계 (인터랙션)
Step 5. CLI 대화 루프      ← 4단계 (완성)
Step 6. 시연 시나리오 검증  ← 5단계
Step 7. 도전 과제          ← 실무 확장
```

## ⚠️ 시작 전 확인

- [ ] OpenAI API 키가 준비되어 있음 (https://platform.openai.com/api-keys)

준비 됐으면 시작합시다!


---

# Step 0. 초기 설정

## 0-1. 패키지 설치

LangChain 관련 패키지를 설치합니다.

- `langchain` : LangChain 메인 패키지
- `langchain-openai` : OpenAI 모델 연동
- `python-dotenv` : `.env` 파일 로드용

> 💡 이미 설치된 환경이라면 그대로 통과되므로 그냥 실행하세요.


In [ ]:
# 패키지 설치
#!uv add  langchain langchain-openai python-dotenv

## 0-2. OpenAI API 키 설정

API 키를 환경변수에 등록합니다. 다음 우선순위로 동작합니다.

1. **로컬 `.env` 파일** — `OPENAI_API_KEY=sk-...` 형태로 적혀 있으면 자동 로드
2. **이미 환경변수에 등록되어 있으면** — 그대로 사용

> ⚠️ API 키는 **절대 코드에 직접 적지 마세요**. 노트북을 공유하면 그대로 노출됩니다.


---

# Step 1. 환경 점검

## 1-1. 필요한 패키지 import

| 도구                 | 역할                           | 어디서 배웠나 |
| -------------------- | ------------------------------ | ------------- |
| `ChatOpenAI`         | LangChain의 OpenAI 모델 클래스 | ① Models      |
| `ChatPromptTemplate` | 프롬프트 템플릿                | ② Prompts     |


In [1]:
# 패키지 import
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

print("✅ import 완료")

✅ import 완료


## 1-2. 2교시 기본 패턴이 동작하는지 확인

오늘 챗봇의 뼈대가 되는 `prompt | llm` 패턴을 한 번 돌려봅시다.

> 💡 만약 여기서 에러가 나면 챗봇을 만들 수 없습니다. 먼저 환경 문제부터 해결하세요.


In [2]:
# 2교시에서 배운 기본 패턴
llm = ChatOllama(model="gemma3:4b", temperature=0.3)

prompt = ChatPromptTemplate.from_messages(
    [("system", "당신은 {role}입니다."), ("human", "{question}")]
)

chain = prompt | llm  # LCEL 파이프 연산자

# 동작 확인
result = chain.invoke(
    {"role": "데이터 분석가", "question": "이상치란 무엇인가요? 한 문장으로."}
)
print(result.content)

이상치는 데이터 세트에서 다른 값들과 현저하게 다른 값으로, 데이터의 일반적인 패턴을 왜곡할 수 있습니다. 

좀 더 자세히 설명하자면, 이상치는 통계적 분석에서 중요한 개념으로, 데이터의 분포를 이해하고 모델을 구축하는 데 영향을 미칠 수 있습니다. 

혹시 이상치에 대해 더 궁금한 점이 있으신가요? 예를 들어, 다음과 같은 질문에 답변해 드릴 수 있습니다.

*   이상치를 어떻게 탐지할 수 있나요?
*   이상치를 처리하는 방법은 무엇인가요?
*   이상치가 분석 결과에 미치는 영향은 무엇인가요?


**✅ 결과 확인**

분석가 톤의 한 문장 답변이 출력되면 환경 OK입니다.

에러가 나면:

- `AuthenticationError` → Step 0-2 다시 실행해서 API 키 재입력
- `ModuleNotFoundError` → Step 0-1 다시 실행 후 커널 재시작


---

# Step 2. 페르소나 설계 (실무 5단계 중 1·2단계)

## 2-1. 왜 페르소나를 먼저 설계하나?

**system 메시지 한 줄로 LLM의 답변 톤이 완전히 바뀝니다.**

실무 챗봇 구축에서 가장 먼저 정해야 하는 것이 바로 "이 챗봇이 누구인가"입니다. 페르소나가 흔들리면 답변 톤이 흔들리고, 답변 톤이 흔들리면 사용자가 떠납니다.

## 2-2. 페르소나 사전 만들기

5가지 페르소나를 Python 사전(dict)으로 정의합니다. **key는 페르소나 이름, value는 system 메시지**예요.


In [3]:
# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
    "마케터": "당신은 디지털 마케팅 전문가입니다. 비즈니스 관점과 사례를 들어 답변하세요.",
    "개발자": "당신은 시니어 백엔드 개발자입니다. 코드 예시와 함께 실용적으로 답변하세요.",
    "기획자": "당신은 IT 서비스 기획자입니다. 사용자 가치와 우선순위 관점에서 답변하세요.",
    "디자이너": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
}

print(f"✅ {len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")

✅ 5개 페르소나 등록 완료: ['분석가', '마케터', '개발자', '기획자', '디자이너']


## 2-3. 페르소나 차이 확인

같은 질문에 페르소나별로 답변이 어떻게 달라지는지 직접 확인해봅시다.


In [4]:
# 같은 질문, 다른 페르소나
question = "좋은 데이터란 무엇인가요?"

for name in ["분석가", "마케터"]:
    result = chain.invoke({"role": PERSONAS[name], "question": question})
    print(f"\n[{name}]")
    print(result.content)
    print("-" * 60)


[분석가]
좋은 데이터란 다음과 같은 특징을 가진 데이터를 의미합니다.

1.  **정확성 (Accuracy):** 데이터가 실제 현상을 정확하게 반영해야 합니다. 오류율은 일반적으로 1% 미만, 이상치(Outlier)는 2-3 표준편차 이내로 관리하는 것이 좋습니다. 
2.  **완전성 (Completeness):** 데이터에 누락된 값이 없어야 합니다. 누락률은 5% 미만, 특정 변수의 누락률은 10% 미만으로 유지하는 것이 이상적입니다.
3.  **일관성 (Consistency):** 데이터가 서로 모순되지 않고, 동일한 정보를 동일한 방식으로 표현해야 합니다. 데이터 타입, 단위, 코드 값 등에 대한 표준화가 중요합니다.
4.  **적시성 (Timeliness):** 데이터가 분석 목적에 필요한 시점에 수집되어야 합니다. 데이터 업데이트 주기는 분석 유형에 따라 다르지만, 일반적으로 실시간 또는 매일 업데이트되는 데이터가 선호됩니다.
5.  **관련성 (Relevance):** 데이터가 분석하고자 하는 질문이나 문제와 관련이 있어야 합니다. 불필요한 데이터는 분석의 정확성을 저해하고, 시간과 비용을 낭비하게 합니다.

**근거:**

*   **데이터 품질 평가:** Gartner 보고서에 따르면, 데이터 품질 문제는 기업의 의사 결정 실패, 운영 비용 증가, 고객 불만 등의 주요 원인으로 작용하며, 데이터 품질 개선에는 평균적으로 6-10%의 비용이 소요될 수 있습니다.
*   **머신러닝 성능:** 머신러닝 모델의 성능은 데이터 품질에 매우 민감합니다. 데이터 품질이 낮은 경우, 모델의 정확도, 정밀도, 재현율 등이 크게 저하될 수 있습니다. (예: Google의 연구에 따르면, 데이터 품질이 1% 향상될 때마다 머신러닝 모델의 정확도가 10% 향상될 수 있습니다.)

이러한 특징들을 종합적으로 고려하여 데이터를 관리하고 평가하는 것이 효과적인 데이터 분석 및 의사 결정에 필수적입니다.
----------------------------------

**✅ 결과 확인**

같은 질문이지만 답변 톤과 관점이 완전히 다른 것을 확인하세요. 이것이 system 메시지의 위력입니다.

> 💡 페르소나 사전을 dict로 둔 이유: 나중에 페르소나 추가/수정이 매우 쉽기 때문이에요. JSON 파일로 분리하면 코드 수정 없이도 페르소나만 갈아끼울 수 있습니다 (도전 과제 참고).


---

# Step 3. 챗봇 함수 만들기 (실무 5단계 중 3단계 - MVP)

## 3-1. 가장 단순한 형태부터

신입은 처음부터 모든 기능을 넣으려 합니다. 시니어는 **"한 번 질문 → 한 번 답변"** 부터 만들어요.

이번 단계의 목표는 **`ask("분석가", "...")` 한 줄로 답변을 받는 함수**를 만드는 것입니다.


In [5]:
# 챗봇 핵심 함수: 페르소나 이름과 질문을 받아 답변 반환
def ask(persona_name: str, question: str) -> str:
    # 1. 페르소나 이름으로 system 메시지 조회
    system_message = PERSONAS[persona_name]

    # 2. 체인 호출 (2교시에서 배운 prompt | llm 패턴)
    result = chain.invoke({"role": system_message, "question": question})

    # 3. AIMessage 객체에서 답변 텍스트만 꺼내서 반환
    return result.content

## 3-2. 함수 테스트

만든 함수를 실제로 호출해봅시다.


In [6]:
# 분석가 페르소나로 질문
answer = ask("분석가", "A/B 테스트를 한 줄로 설명해주세요")
print(answer)

A/B 테스트는 웹사이트, 앱, 마케팅 캠페인 등에서 두 가지 버전(A와 B)을 비교하여 어떤 버전이 더 나은 성과를 내는지 측정하는 방법입니다. 일반적으로 두 버전 중 하나를 무작위로 선택하여 사용자에게 보여주고, 클릭률, 전환율 등 주요 지표를 측정하여 통계적으로 유의미한 차이가 있는지 확인합니다. 

**근거:**

*   **통계적 유의성:** A/B 테스트는 충분한 데이터 수집을 통해 얻은 결과를 바탕으로, 우연에 의한 차이(Type I error)를 최소화하고 실제 효과를 정확하게 파악합니다. 일반적으로 p-value가 0.05(5%) 이하이면 통계적으로 유의미한 결과로 간주됩니다.
*   **데이터 기반 의사 결정:** A/B 테스트는 직관이나 추측이 아닌 실제 사용자 데이터를 기반으로 의사 결정을 내릴 수 있도록 합니다.
*   **지속적인 개선:** A/B 테스트는 지속적으로 데이터를 수집하고 분석하여 제품 또는 서비스의 성능을 개선하는 데 활용됩니다.

**예시:**

*   버전 A: 버튼 색상을 파란색으로
*   버전 B: 버튼 색상을 빨간색으로

두 버전의 클릭률을 비교하여 어떤 색상이 더 많은 클릭을 유도하는지 확인합니다.


**✅ 결과 확인**

분석가 톤의 한두 문장 답변이 나오면 성공!

이제 챗봇의 **두뇌 부분**은 완성되었어요. 이 함수 하나만 있으면 챗봇 로직은 끝난 거예요. 나머지는 다 입출력 처리입니다.


---

# Step 4. 명령어 처리 (실무 5단계 중 4단계)

## 4-1. 사용자 입력의 두 종류

CLI 챗봇 사용자는 두 가지를 입력합니다.

| 입력                  | 의미              | 처리 방법    |
| --------------------- | ----------------- | ------------ |
| 일반 텍스트           | LLM에게 보낼 질문 | `ask()` 호출 |
| `/`로 시작하는 텍스트 | 명령어            | 별도 처리    |

명령어를 처리하는 함수를 따로 만들어봅시다. 챗봇 로직(`ask`)과 명령어 로직을 분리하면 코드가 깔끔해져요.

## 4-2. 명령어 처리 함수

이 함수는 다음 명령어를 처리합니다.

- `/quit`, `/exit` → 종료 신호 반환
- `/change 마케터` → 페르소나 변경
- `/persona` → 현재 페르소나 확인
- `/help` → 도움말


In [7]:
# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)  # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True  # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False  # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

## 4-3. 명령어 처리 테스트

대화 루프를 만들기 전에 명령어가 잘 동작하는지 확인합시다.


In [8]:
# 페르소나 변경 시뮬레이션
current = "분석가"
current, _ = handle_command("/persona", current)
current, _ = handle_command("/change 마케터", current)
current, _ = handle_command("/persona", current)
current, _ = handle_command("/change 외계인", current)  # 잘못된 페르소나
current, _ = handle_command("/help", current)

🎭 현재 페르소나: 분석가
✅ 페르소나가 '마케터'(으)로 변경되었습니다.
🎭 현재 페르소나: 마케터
❌ '외계인'은 등록된 페르소나가 아닙니다.
   사용 가능: 분석가, 마케터, 개발자, 기획자, 디자이너
📖 명령어: /quit, /exit, /change <이름>, /persona, /help


**✅ 결과 확인**

- `/persona`로 현재 페르소나 확인됨
- `/change 마케터`로 변경됨
- `/change 외계인` 같은 잘못된 이름은 거절됨
- `/help`로 도움말이 출력됨

이 4가지가 모두 동작하면 명령어 처리는 완성!


---

# Step 5. CLI 대화 루프 (챗봇 완성!)

## 5-1. 대화 루프란?

지금까지 만든 부품은 3개입니다.

```
PERSONAS          ← 페르소나 사전 (Step 2)
ask()             ← 챗봇 두뇌 (Step 3)
handle_command()  ← 명령어 처리 (Step 4)
```

이걸 모두 묶어서 **"입력을 기다리고 → 처리하고 → 다시 입력 기다리는"** 무한 루프를 만들면 챗봇이 완성됩니다.

## 5-2. 챗봇 실행 함수


In [ ]:
# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:  # 빈 입력은 무시
            continue

        if user_input.startswith("/"):  # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:  # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")
            print()
            print("===" * 25)

## 5-3. 챗봇 실행!

아래 셀을 실행하면 CLI 챗봇이 시작됩니다.

> ⚠️ **Jupyter에서 `input()`은 노트북 상단에 입력창이 뜹니다.** 입력 후 Enter를 치세요.
>
> 종료하려면 `/quit` 또는 `/exit` 입력.


In [10]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

[분석가] 안녕하세요. 10년차 데이터 분석가입니다. 어떤 질문이든 간결하고 수치, 근거를 기반으로 답변 드리겠습니다. 무엇을 도와드릴까요?

[분석가] openai API 사용 시 예상치 못한 토큰 사용량 증가 및 과다 청구 문제는 여러 요인으로 발생할 수 있습니다. 10년차 데이터 분석가로서, 다음과 같은 가능성을 고려하고 수치적 근거를 들어 설명드리겠습니다.

**1. 프롬프트 엔지니어링 오류:**

*   **평균 프롬프트 길이:** 일반적으로 openai 모델은 프롬프트의 길이가 길수록 토큰 사용량이 증가합니다. 평균 프롬프트 길이가 200 토큰 수준이라면, 9백만 토큰은 매우 비정상적인 수준입니다.
*   **반복적인 프롬프트:** 모델에게 동일한 질문을 반복적으로 던지거나, 프롬프트 내에 불필요한 반복문을 포함하면 토큰 사용량이 급증할 수 있습니다.
*   **데이터 포함:** 프롬프트에 과도한 양의 데이터(예: 긴 문서, 이미지 설명 등)를 포함하면 토큰 사용량이 늘어납니다. 

**2. 모델 설정 오류:**

*   **최대 토큰 설정:** API 호출 시 설정한 최대 토큰 수가 너무 높으면 모델이 생성하는 텍스트의 양에 제한 없이 증가하여 토큰 사용량이 폭증할 수 있습니다.
*   **모델 선택:** 모델별로 토큰 사용 효율성이 다릅니다. 예를 들어, GPT-4는 GPT-3.5보다 더 많은 토큰을 사용하지만, 더 높은 품질의 결과를 제공할 수 있습니다.

**3. 예상치 못한 사용 패턴:**

*   **자동화된 스크립트:** 자동화된 스크립트나 애플리케이션을 통해 API를 호출하는 경우, 예상치 못한 조건에서 토큰 사용량이 급증할 수 있습니다.
*   **실험적 사용:** 새로운 프롬프트나 모델을 실험적으로 사용하는 경우, 토큰 사용 패턴을 제대로 파악하지 못해 과다 청구될 수 있습니다.

**4. 시스템 오류:**

*   **openai 서

---

# Step 6. 시연 시나리오 검증 (실무 5단계 중 5단계)

가이드 문서의 **5개 시연 시나리오**를 직접 통과시켜봅시다. 위의 `run_chatbot()` 셀을 실행한 뒤 아래 입력을 순서대로 넣어보세요.

## 시나리오 체크리스트

```
[ ] 시나리오 1: 일반 질문 → 분석가 톤 답변
    질문> 머신러닝이 뭐예요?

[ ] 시나리오 2: 페르소나 변경 후 같은 질문 → 답변 톤이 바뀜
    질문> /change 마케터
    질문> 머신러닝이 뭐예요?

[ ] 시나리오 3: 잘못된 페르소나 → 거절
    질문> /change 외계인

[ ] 시나리오 4: 도움말/상태 조회
    질문> /persona
    질문> /help

[ ] 시나리오 5: 정상 종료
    질문> /quit
```


---

# Step 7. 도전 과제 (실무 확장)

기본 실습이 끝났다면, 실무에서 챗봇을 만들 때 자주 마주치는 확장 작업을 시도해보세요. **코드 난이도를 높이는 게 아니라**, 실무에서 첫 번째 챗봇 빌드 후 바로 따라오는 작업들입니다.

## 🎯 과제 1: 대화 로그 저장 (실무 빈도 ★★★★★)

운영 환경에서는 사용자가 무엇을 묻고 챗봇이 무엇을 답했는지 **반드시** 로그로 남깁니다. 디버깅·품질 개선·법적 대응 모든 면에서 필요해요.

**요구사항**
- 챗봇이 종료될 때 `chat_log.txt` 파일에 대화 내용 저장
- 형식: `[시각] [페르소나] 질문 → 답변`

**힌트**
```python
from datetime import datetime
log = []   # 대화 누적 리스트
log.append(f"[{datetime.now().strftime('%H:%M:%S')}] [{persona}] {user_input} → {answer}")
# 종료 시:
with open("chat_log.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(log))
```

---

## 🎯 과제 2: 페르소나별 temperature 다르게 설정 (실무 빈도 ★★★★☆)

1교시에서 배웠죠. `temperature`는 페르소나의 성격과도 연결됩니다.

- **분석가**: `temperature=0` (정확하고 일관된 답변)
- **디자이너**: `temperature=0.8` (창의적이고 다양한 답변)

**요구사항**

- `PERSONAS` 사전 구조를 `{name: {"system": ..., "temperature": ...}}` 형태로 확장
- `ask()` 함수에서 페르소나별 `temperature`로 호출
- **힌트**: `ChatOpenAI(model=..., temperature=...)`을 페르소나마다 새로 만들거나, `.bind(temperature=...)`을 사용

---

## 🎯 과제 3: 페르소나를 JSON 파일로 분리 (실무 빈도 ★★★★☆)

실무에서는 페르소나가 자주 추가·수정됩니다. 코드 안에 박혀 있으면 매번 코드를 고쳐야 해요. **외부 파일로 분리하면 비개발자도 페르소나를 편집할 수 있습니다.**

**요구사항**

- `personas.json` 파일에 페르소나 정의 저장
- 챗봇 시작 시 JSON을 읽어와 사용

**힌트**

```python
import json
with open("personas.json", encoding="utf-8") as f:
    PERSONAS = json.load(f)
```

---

## 🎯 과제 4: `/list` 명령어 추가 (실무 빈도 ★★★☆☆)

사용자가 어떤 페르소나가 있는지 매번 기억할 수 없어요. 목록 조회 명령어를 추가하세요.

**요구사항**

- `/list` 입력 시 페르소나 이름과 한 줄 소개 출력
- 예: `- 분석가: 수치와 근거 기반 답변`

---

## 🎯 과제 5 (선택): max_tokens로 답변 길이 제어 (실무 빈도 ★★★☆☆)

1교시에서 배운 `max_tokens` 파라미터를 적용해보세요. 토큰 폭주 방지와 비용 관리에 핵심입니다.

**요구사항**

- 모든 답변을 200 토큰 이내로 제한
- **힌트**: `ChatOpenAI(model="gpt-4o-mini", temperature=0.3, max_tokens=200)`

---

> 💡 **5개 모두 할 필요 없습니다.** 본인이 실무에서 가장 가깝다고 느끼는 1-2개만 시도해보세요. 시간이 남으면 추가로 도전!


---

### 실무 사고법

- 챗봇 구축 5단계: **요구사항 → 페르소나 → MVP → 인터랙션 → 검증**
- 가장 단순한 형태(MVP)부터 만들고 단계적으로 확장
- 로직과 UI는 분리

### 코드 패턴

- `prompt | llm` LCEL 체인을 함수로 감싸 재사용
- 페르소나를 사전(dict)으로 관리
- 명령어 분기를 별도 함수로 분리
- CLI 입력 루프 (`while True` + `input()`)
